# F1a — LiDAR-Aug: trung bình reward qua 4 biến đổi ảnh (SD 1.5, DDPM-100, seed 100)

**Notebook này chỉ chạy một run: LiDAR-Aug.** Thời gian (L4): chấm lại khoảng 1–1.5 h (decode 50 ứng viên, 200 lần gọi IR mỗi prompt), Phase 2 khoảng 3–5 h, GenEval khoảng 30 phút. Tổng khoảng **5–7 h**.

**Mục đích.** Reviewer sẽ hỏi: *"Sao phải làm trơn bằng nhiễu Gaussian trên pixel? Lấy trung bình reward theo cách khác có được không?"* F1 so RS-LiDAR với hai cách khác, đều gọi ImageReward **4 lần cho mỗi ứng viên**, như RS-LiDAR (M = 4):
- **LiDAR-Aug:** trung bình IR qua 4 biến đổi nhẹ, cố định: lật ngang, crop 90% rồi resize lại, JPEG q = 75, Gaussian blur r = 1;
- **LiDAR-LatentSmooth:** cộng nhiễu N(0, 0.3²) vào latent của ứng viên, decode, chấm IR, lấy trung bình 4 lần.

Cả hai chấm lại **đúng 50 ứng viên** của pool seed 100 (Table 1), rồi chạy Phase 2 với seed 100 như Table 1, nên so được ghép cặp với run mốc LiDAR và RS-LiDAR seed 100 đã có (`E1_…`). Không sinh ứng viên mới, không sửa code trong repo.

**Cách chạy:** chọn **L4 + High-RAM**, rồi **Runtime → Run all**. Không sửa gì: notebook đã cấu hình sẵn. Phần chấm (mục 7) chạy tiếp được sau khi bị ngắt. Phase 2 (mục 8) dùng seed toàn cục 100 nên phải chạy **một mạch**; nếu bị ngắt giữa Phase 2, notebook in đường dẫn và dừng: xoá thư mục `F1_…` đó trên Drive rồi Run all lại.

**Kết quả** (`MyDrive/RS-LiDAR/extra_experiments/F1_augmentation/`), tính trên mọi run F1 đã có trên Drive:
- `F1_metrics.md`: bảng gọn (IR, CLIP, HPS, GenEval; không CI) kèm kết luận;
- `F1_summary.csv`: có CI 95% (Δ = run − RS-LiDAR).

## 1. Kiểm Tra Phần Cứng GPU & VRAM
Đảm bảo bạn đã chọn Runtime: **Runtime -> Change runtime type -> A100 (hoặc L4) GPU** và bật **High-RAM**.

In [ ]:
import torch, sys, os
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f"🎮 GPU: {gpu_name}")
    print(f"💾 VRAM: {vram_gb} GB")
    if vram_gb >= 22.0:
        print("✅ Tuyệt vời! GPU có đủ VRAM để kích hoạt Full-Batch VAE Decoding tốc độ cao.")
    else:
        print("⚠️ Chú ý: GPU dưới 22GB VRAM, code sẽ tự động dùng chunking để bảo vệ bộ nhớ.")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU.")

!nvidia-smi

## 2. Gắn Kết Google Drive & Thiết Lập Lưu Trữ Tức Thì (Real-time Persistence)
Toàn bộ kết quả sẽ được ghi trực tiếp vào `My Drive/RS-LiDAR/` thông qua symlink. Sau mỗi prompt hoàn thành, file `.png`, `latent.pt` và `results.json` được lưu vĩnh viễn trên Drive.

In [ ]:
import os, shutil, subprocess
from google.colab import drive

# 1. Gắn kết Google Drive an toàn
drive.mount('/content/drive')

base_drive = "/content/drive/My Drive" if os.path.exists("/content/drive/My Drive") else "/content/drive/MyDrive"
assert os.path.exists(base_drive), f"❌ Không tìm thấy thư mục Google Drive tại {base_drive}! Vui lòng gắn kết lại Drive."

DRIVE_DIR = f"{base_drive}/RS-LiDAR"
os.makedirs(f"{DRIVE_DIR}/Lookahead_samples", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/Target_samples", exist_ok=True)

# 2. Clone hoặc kéo cập nhật repo RS-LiDAR tự động
REPO_DIR = "/content/RS-LiDAR"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/leekwanreal/RS-LiDAR.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

WORKDIR = REPO_DIR if os.path.exists(f"{REPO_DIR}/lookahead_sampling.py") else f"{REPO_DIR}/Diffusion-LiDAR-Sampling"
%cd {WORKDIR}

# 3. Tạo Symbolic Links an toàn (Tuyệt đối không xóa dữ liệu cũ, tự động migrate nếu có file cục bộ)
for folder_name in ["Lookahead_samples", "Target_samples"]:
    link_path = os.path.join(WORKDIR, folder_name)
    drive_target = os.path.join(DRIVE_DIR, folder_name)
    os.makedirs(drive_target, exist_ok=True)

    if os.path.islink(link_path):
        if not os.path.exists(link_path):
            os.remove(link_path)
            os.symlink(drive_target, link_path)
    elif os.path.exists(link_path):
        print(f"📦 Phát hiện thư mục cục bộ {folder_name}, đang đồng bộ an toàn sang Google Drive...")
        for item in os.listdir(link_path):
            src_item = os.path.join(link_path, item)
            dst_item = os.path.join(drive_target, item)
            if not os.path.exists(dst_item):
                if os.path.isdir(src_item):
                    shutil.copytree(src_item, dst_item)
                else:
                    shutil.copy2(src_item, dst_item)
        shutil.rmtree(link_path)
        os.symlink(drive_target, link_path)
    else:
        os.symlink(drive_target, link_path)

print("✅ Hệ thống Symlink đã sẵn sàng 100%! Toàn bộ dữ liệu được bảo lưu vĩnh viễn trên Drive:")
print("  📁", DRIVE_DIR)

## 3. Cài Đặt Môi Trường Chuẩn Xác (100% Trơn Tru & Không Xung Đột)
Cài đặt bộ thư viện đồng bộ, tắt backend TensorFlow để chống xung đột Protobuf, và tải vocab HPSv2.

In [ ]:
import os, urllib.request

# 1. Tắt backend TensorFlow để loại bỏ xung đột Protobuf runtime_version
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

# 2. Nâng cấp protobuf và cài đặt các thư viện tương thích PyTorch 2.x
!pip install -q --upgrade protobuf open_clip_torch
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q image-reward || pip install -q --no-deps git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

# 3. Đảm bảo file từ điển OpenCLIP cho HPSv2
import hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    print("📥 Đang tải file BPE vocab cho HPSv2...")
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)

print("✅ Cài đặt môi trường hoàn tất 100%!")

In [ ]:
# Kiểm tra môi trường sau khi cài: in phiên bản và dừng ngay nếu thiếu thư viện chấm điểm
# (thiếu hpsv2/clip thì code reward giờ báo lỗi thay vì chấm 0; dừng ở đây đỡ tốn GPU hơn).
import importlib, sys
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
from rslidar_utils import setup_runtime
setup_runtime()  # cùng các shim tương thích mà script của repo áp dụng trước khi import ImageReward
import torch, numpy, transformers, diffusers
print("python", sys.version.split()[0], "| torch", torch.__version__, "| numpy", numpy.__version__,
      "| transformers", transformers.__version__, "| diffusers", diffusers.__version__)
for mod in ("hpsv2", "clip", "open_clip", "ImageReward", "timm", "google.protobuf"):
    try:
        m = importlib.import_module(mod)
        print(f"  ✅ {mod} {getattr(m, '__version__', '')}")
    except Exception as e:
        raise RuntimeError(f"❌ Không import được {mod}: {e}. Chạy lại cell cài đặt (hoặc Runtime -> Restart session rồi chạy lại).")

## 4. Cấu hình chung (giống Table 1) và hàm tiện ích

In [ ]:
import os, sys, glob, json, shutil, subprocess, random, math
import numpy as np, pandas as pd
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
%cd {WORKDIR}
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

EXP_NAME = "F1_augmentation"
EXP_DRIVE = f"{DRIVE_DIR}/extra_experiments/{EXP_NAME}"      # bảng tổng hợp CSV/JSON lưu vĩnh viễn trên Drive
LOCAL_DERIVED = f"/content/derived_lookahead/{EXP_NAME}"    # lookahead dẫn xuất: tạo lại được, để trên đĩa Colab
os.makedirs(EXP_DRIVE, exist_ok=True)
os.makedirs(LOCAL_DERIVED, exist_ok=True)

# ---- Cấu hình Table 1: SD 1.5, lookahead DPM-5 với K = 50, target DDPM-100, seed 100 ----
MODEL_NAME = "runwayml/stable-diffusion-v1-5"
MODEL_TAG = "SD15"
STEPS_P1, K = 5, 50
SEED_P1, SEED_P2 = 100, 100
SCALE, STEPS_P2, ETA_P2 = 12.5, 100, 1.0
NUM_PARTICLES_P2, RESAMPLE_T_END = 4, 200
REF_LAMBDA = 5000
RS_SIGMA, RS_M = 1.0, 4          # RS-LiDAR DDPM của Table 1
RS_USE_CRN = False               # bài ghi "Perturbations: i.i.d." -> chỉ nhận pool/run I.I.D. (tên không có _crn)

PROMPT_MODE = "ALL"   # "ALL" (553 prompt) hoặc "100_REPRESENTATIVE" (chạy thử nhanh)
if PROMPT_MODE == "100_REPRESENTATIVE":
    MAX_PROMPT, PROMPT_INDICES_ARG, SUBSET_TAG = 100, "stratified_100", "_100p"
else:
    MAX_PROMPT, PROMPT_INDICES_ARG, SUBSET_TAG = 553, None, ""

from rslidar_utils import parse_prompt_indices
META = [json.loads(l) for l in open(f"{WORKDIR}/prompt_files/geneval_metadata.jsonl", encoding="utf-8") if l.strip()]
PROMPT_IDS = list(parse_prompt_indices(PROMPT_INDICES_ARG) or range(len(META)))[:MAX_PROMPT]
print(f"{EXP_NAME}: {len(PROMPT_IDS)} prompt | kết quả tổng hợp lưu tại {EXP_DRIVE}")

In [ ]:
import torch
from tqdm.auto import tqdm
from PIL import Image

STEP_VARIANTS = [f"Step{STEPS_P1}", f"DPM{STEPS_P1}", f"DPM{STEPS_P1}_Step{STEPS_P1}"]


def n_have(d, ids=None):
    """Số prompt trong PROMPT_IDS đã có results.json trong thư mục d."""
    ids = PROMPT_IDS if ids is None else ids
    if not d or not os.path.isdir(d):
        return 0
    return sum(os.path.exists(f"{d}/{i:05d}/results.json") for i in ids)


def find_lookahead(names):
    """Thư mục Lookahead đầu tiên phủ đủ PROMPT_IDS (thư mục 553 prompt cũng dùng được cho tập 100)."""
    for name in names:
        for suf in dict.fromkeys([SUBSET_TAG, ""]):
            p = f"{WORKDIR}/Lookahead_samples/{name}{suf}"
            if n_have(p) >= len(PROMPT_IDS):
                return p
    return None


def find_target(names):
    """Thư mục Target theo quy ước tên của notebook Table 1 (..._seed100[_100p]_A100)."""
    for name in names:
        # run mốc phải cùng tập prompt: seed toàn cục nối tiếp qua các prompt, nên run 553 prompt không ghép cặp với tập 100
        for suf in ([SUBSET_TAG] if SUBSET_TAG else [""]):
            for p in (f"{WORKDIR}/Target_samples/{name}{suf}_A100", f"{WORKDIR}/Target_samples/{name}{suf}"):
                if n_have(p) >= len(PROMPT_IDS):
                    return p
    return None


def raw_lookahead_names(seed=SEED_P1):
    names = [f"Lookahead_{MODEL_TAG}_{s}_n{K}_seed{seed}" for s in STEP_VARIANTS]
    names.append(f"{seed}_{K}_{STEPS_P1}")
    if seed == 100:
        names.append(f"100_{K}_{STEPS_P1}")
    return names


def rs_lookahead_names(sigma=RS_SIGMA, M=RS_M):
    crns = ["", "_crn"] if RS_USE_CRN is None else (["_crn"] if RS_USE_CRN else [""])
    return [f"RSLookahead_{MODEL_TAG}_{s}_n{K}_sig{sigma}_M{M}{c}_seed{SEED_P1}" for c in crns for s in STEP_VARIANTS]


def ref_target_names(rs):
    solver = f"DDPM{STEPS_P2}"
    if rs:
        crns = ["", "_crn"] if RS_USE_CRN is None else (["_crn"] if RS_USE_CRN else [""])
        return [f"RSLiDAR_{MODEL_TAG}_{s}_n{K}_sig{RS_SIGMA}_M{RS_M}{c}_{solver}_s{SCALE}_lmbda{REF_LAMBDA}_seed{SEED_P2}"
                for c in crns for s in STEP_VARIANTS]
    return [f"LiDAR_{MODEL_TAG}_{s}_n{K}_{solver}_s{SCALE}_lmbda{REF_LAMBDA}_seed{SEED_P2}" for s in STEP_VARIANTS]


def stream(cmd):
    print("$ " + " \\\n    ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=WORKDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Lệnh thất bại (exit {proc.returncode}): {cmd[1]}")


def phase1(run_name, seed=SEED_P1, reuse_from=None, smoothing=False, sigma=RS_SIGMA, M=RS_M, crn=False,
           metrics="ImageReward"):
    """Phase 1 (lookahead_sampling.py) với đúng các cờ của notebook Table 1; bỏ qua nếu đã đủ."""
    out = f"{WORKDIR}/Lookahead_samples/{run_name}"
    if n_have(out) >= len(PROMPT_IDS):
        print(f"✅ Phase 1 đã đủ {len(PROMPT_IDS)} prompt: {run_name}")
        return out
    cmd = ["python", "lookahead_sampling.py", f"--seed={seed}", f"--num_particles={K}",
           f"--num_inference_steps={STEPS_P1}", f"--model_name={MODEL_NAME}",
           "--guidance_reward_fn=ImageReward", f"--metrics_to_compute={metrics}",
           "--prompt_path=prompt_files/geneval_metadata.jsonl", f"--max_prompt={MAX_PROMPT}",
           f"--run_name={run_name}", "--resume"]
    if PROMPT_INDICES_ARG:
        cmd.append(f"--prompt_indices={PROMPT_INDICES_ARG}")
    if reuse_from:
        cmd.append(f"--reuse_latents_from={reuse_from}")
    if smoothing:
        cmd += ["--use_smoothing", f"--sigma={sigma}", f"--num_mc_samples={M}"]
        if crn:
            cmd.append("--use_crn")
    stream(cmd)
    return out


def phase2(lookahead_dir, run_name, lmbda=REF_LAMBDA, top_k=K, pps=False):
    """Phase 2 (LiDAR_sampling.py) với đúng các cờ của notebook Table 1; bỏ qua nếu đã đủ.

    pps=True thêm --per_prompt_seed (nhiễu theo prompt: chạy tiếp sau khi bị ngắt vẫn đúng cặp).
    pps=False (so với run Table 1, seed toàn cục): run dở dang bị chặn, phải xoá và chạy lại một mạch.
    """
    out = f"{WORKDIR}/Target_samples/{run_name}"
    have = n_have(out)
    if have >= len(PROMPT_IDS):
        print(f"✅ Phase 2 đã đủ {len(PROMPT_IDS)} prompt: {run_name}")
        return out
    if have > 0 and not pps:
        raise RuntimeError(f"Run dở dang ({have}/{len(PROMPT_IDS)} prompt): {out}\n"
                           "Run này dùng seed toàn cục nên chạy tiếp sẽ lệch nhiễu so với run đối chứng. "
                           "Xoá thư mục trên rồi chạy lại cell này.")
    cmd = ["python", "LiDAR_sampling.py", f"--seed={SEED_P2}", f"--model_name={MODEL_NAME}",
           f"--num_particles={NUM_PARTICLES_P2}", f"--num_inference_steps={STEPS_P2}", f"--eta={ETA_P2}",
           "--use_rag", f"--lookahead_path={lookahead_dir}", f"--top_k={top_k}", f"--scale={SCALE}",
           f"--lmbda={lmbda}", f"--resample_t_end={RESAMPLE_T_END}",
           "--prompt_path=prompt_files/geneval_metadata.jsonl", f"--max_prompt={MAX_PROMPT}",
           "--guidance_reward_fn=ImageReward",
           "--metrics_to_compute=ImageReward#Clip-Score#Clip-Diversity#HumanPreference#AS",
           "--save_individual_images", f"--run_name={run_name}", "--resume"]
    if PROMPT_INDICES_ARG:
        cmd.append(f"--prompt_indices={PROMPT_INDICES_ARG}")
    if pps:
        cmd.append("--per_prompt_seed")
    stream(cmd)
    return out


def pools_paired(a, b, n_check=3):
    """True if pools a and b hold the same latents (up to order) for the first n_check prompts."""
    for i in PROMPT_IDS[:n_check]:
        if not (os.path.exists(f"{a}/{i:05d}/samples/latent.pt") and os.path.exists(f"{b}/{i:05d}/samples/latent.pt")):
            return False
        La, _ = load_pool(a, i)
        Lb, _ = load_pool(b, i)
        if La.shape != Lb.shape:
            return False
        A, Bm = La.flatten(1).float(), Lb.flatten(1).float()
        d = torch.cdist(Bm, A, compute_mode="donot_use_mm_for_euclid_dist").min(1).values
        if (d > 1e-3 * A.norm(dim=1).mean()).any():
            return False
    return True


def paired_rs_pool(raw_dir, rs_dir):
    """RS pool scored on exactly raw_dir's latents. If the pool found on Drive was generated separately
    (Phase 1 without --reuse_latents_from draws different candidates), rescore raw_dir's latents instead."""
    if rs_dir and pools_paired(raw_dir, rs_dir):
        print(f"✅ Pool RS dùng chung latents với pool LiDAR: {os.path.basename(rs_dir)}")
        return rs_dir
    if rs_dir:
        print(f"⚠️ Pool RS '{os.path.basename(rs_dir)}' KHÔNG dùng chung latents với pool LiDAR -> chấm lại pool LiDAR bằng R̄σ.")
    name = f"RSLookahead_{MODEL_TAG}_Step{STEPS_P1}_n{K}_sig{RS_SIGMA}_M{RS_M}_seed{SEED_P1}_rescore{SUBSET_TAG}"
    return phase1(name, reuse_from=raw_dir, smoothing=True)


def load_pool(d, i, key="ImageReward"):
    """Latents (K, 4, 64, 64) và reward (K,) của prompt i trong thư mục lookahead d."""
    lat = torch.load(f"{d}/{i:05d}/samples/latent.pt", map_location="cpu")
    lat = torch.stack(list(lat)) if isinstance(lat, (list, tuple)) else lat
    res = json.load(open(f"{d}/{i:05d}/results.json"))
    return lat, np.asarray(res[key]["result"], dtype=np.float64)


def _summ(v):
    v = np.asarray(v, dtype=np.float64)
    return {"result": v.tolist(), "mean": float(v.mean()), "std": float(v.std(ddof=1)) if len(v) > 1 else 0.0,
            "max": float(v.max()), "min": float(v.min())}


def write_derived(src_dir, dst_dir, transform):
    """Tạo thư mục lookahead dẫn xuất: cùng latents (symlink), reward mới = transform(i, results).

    LiDAR_sampling.py đọc reward từ khoá "ImageReward" của results.json, nên reward mới được ghi vào
    khoá đó; reward gốc giữ ở "source_results" để đối chiếu. Thứ tự latents không đổi.
    """
    n = 0
    for i in PROMPT_IDS:
        s, d = f"{src_dir}/{i:05d}", f"{dst_dir}/{i:05d}"
        if not os.path.exists(f"{s}/results.json"):
            continue
        res = json.load(open(f"{s}/results.json"))
        new = np.asarray(transform(i, res), dtype=np.float64)
        os.makedirs(f"{d}/samples", exist_ok=True)
        lat_dst = f"{d}/samples/latent.pt"
        if not os.path.lexists(lat_dst):
            os.symlink(f"{s}/samples/latent.pt", lat_dst)
        out = {"ImageReward": _summ(new), "prompt": res["prompt"], "prompt_index": i,
               "source_results": {k: v for k, v in res.items() if isinstance(v, dict) and "result" in v}}
        json.dump(out, open(f"{d}/results.json", "w"))
        n += 1
    print(f"📁 {os.path.basename(dst_dir)}: {n} prompt")
    return dst_dir


def rs_pool_ok(d, n_check=3):
    """Pool RS chỉ dùng được nếu được chấm đúng cấu hình bài: σ, M, nhiễu i.i.d., β = 0, làm trơn trên ảnh.

    Tên thư mục không ghi cách lấy nhiễu, nên phải đọc results.json["smoothing"]. Metadata có ba đời
    (rslidar_utils.read_smoothing_config): trước 27/10/2026 chỉ có use_crn (nhiễu i.i.d. hoặc CRN, không có
    entropic risk); 27/10 - 9/11 mặc định antithetic + β = 0.5; từ 9/11 mặc định i.i.d., β = 0.
    """
    from rslidar_utils import read_smoothing_config
    want = {"sigma": float(RS_SIGMA), "num_mc_samples": int(RS_M), "noise_mode": "independent_iid",
            "beta_risk": 0.0, "domain": "image"}
    legacy = False
    for i in PROMPT_IDS[:n_check]:
        f = f"{d}/{i:05d}/results.json"
        if not os.path.exists(f):
            return False
        sm = read_smoothing_config(json.load(open(f)))
        if not sm:
            print(f"⚠️ {os.path.basename(d)}: results.json không ghi cấu hình smoothing -> không dùng pool này.")
            return False
        bad = {k: (sm[k], v) for k, v in want.items()
               if (sm[k] != v if isinstance(v, str) else abs(float(sm[k]) - v) > 1e-9)}
        if bad:
            print(f"⚠️ {os.path.basename(d)}: cấu hình smoothing khác bài {bad} (có, cần) -> không dùng pool này.")
            return False
        legacy = legacy or sm["legacy_metadata"]
    if legacy:
        print(f"ℹ️ {os.path.basename(d)}: metadata đời cũ (trước 27/10/2026, chỉ có use_crn = False) "
              "-> nhiễu i.i.d., β = 0, đúng cấu hình bài.")
    return True


def ref_ok(target, pool, rs=False, lmbda=REF_LAMBDA, top_k=K):
    """Run mốc (Table 1) chỉ dùng được nếu args.json khớp cấu hình ghép cặp: cùng pool ứng viên, seed toàn cục,
    cùng λ, K, số bước, scale. Không khớp thì trả về False để notebook tự chạy lại mốc."""
    if not target:
        return False
    name = os.path.basename(os.path.normpath(target))
    f = f"{target}/args.json"
    if not os.path.exists(f):
        print(f"⚠️ {name}: không có args.json, không kiểm tra được cấu hình -> vẫn dùng.")
        return True
    a = json.load(open(f))
    want = {"seed": SEED_P2, "lmbda": lmbda, "top_k": top_k, "num_inference_steps": STEPS_P2,
            "num_particles": NUM_PARTICLES_P2, "scale": SCALE, "eta": ETA_P2}
    bad = {k: (a.get(k), v) for k, v in want.items() if k in a and abs(float(a[k]) - float(v)) > 1e-9}
    if a.get("per_prompt_seed"):
        bad["per_prompt_seed"] = (True, False)
    lp = str(a.get("lookahead_path", ""))
    ref_pool = (lp if os.path.isabs(lp) and os.path.exists(lp) else
                f"{WORKDIR}/Lookahead_samples/{os.path.basename(os.path.normpath(lp))}") if lp else None
    if rs and ref_pool and os.path.exists(ref_pool) and not rs_pool_ok(ref_pool):
        bad["smoothing của pool"] = (os.path.basename(os.path.normpath(ref_pool)), "i.i.d., σ, M như bài")
    if pool and ref_pool:
        same = os.path.basename(os.path.normpath(ref_pool)) == os.path.basename(os.path.normpath(pool))
        if not same:
            same = os.path.exists(ref_pool) and pools_paired(ref_pool, pool)
        if not same:
            bad["lookahead_path"] = (os.path.basename(os.path.normpath(lp)), os.path.basename(os.path.normpath(pool)))
    if bad:
        print(f"⚠️ Run mốc {name} không khớp cấu hình {bad} (có, cần) -> không dùng, notebook tự chạy lại mốc.")
        return False
    print(f"✅ Run mốc khớp cấu hình: {name}")
    return True

## 5. Cấu hình của notebook này (đã đặt sẵn, không sửa)

In [ ]:
VARIANT = "aug"
RUN_TAG = "Aug"
RUN_LABEL = "LiDAR-Aug (M=4)"
M_F1 = 4
SIGMA_Z = None
F1_SEED = 2026

## 6. Pool seed 100 của Table 1 và hai run mốc (LiDAR, RS-LiDAR λ = 5000)

In [ ]:
RAW_DIR = find_lookahead(raw_lookahead_names())
assert RAW_DIR, "Không tìm thấy pool lookahead LiDAR seed 100 (Table 1)."
print("✅ Pool:", os.path.basename(RAW_DIR))

# Run mốc: run Table 1 (tên gốc), rồi run mốc E1_… của E2/E1, rồi run mốc E3 (E3_…_K50). Run nào cũng phải qua ref_ok.
REF_NAMES = {
    False: [f"E1_LiDAR_{MODEL_TAG}_n{K}_DDPM100_lmbda{REF_LAMBDA}_seed{SEED_P2}",
            f"E3_LiDAR_{MODEL_TAG}_K50_DDPM100_lmbda{REF_LAMBDA}_seed{SEED_P2}"],
    True: [f"E1_RSLiDAR_{MODEL_TAG}_n{K}_sig{RS_SIGMA}_M{RS_M}_DDPM100_lmbda{REF_LAMBDA}_seed{SEED_P2}",
           f"E3_RSLiDAR_{MODEL_TAG}_K50_M{RS_M}_DDPM100_lmbda{REF_LAMBDA}_seed{SEED_P2}"],
}


def pick_ref(rs):
    for names in [ref_target_names(rs=rs)] + [[n] for n in REF_NAMES[rs]]:
        d = find_target(names)
        if d and ref_ok(d, RAW_DIR, rs=rs):
            return d
    return None


RUNS = {"RS-LiDAR K=50 M=4": pick_ref(rs=True), "LiDAR K=50": pick_ref(rs=False)}
print(RUNS)
assert all(RUNS.values()), "Thiếu run mốc seed 100 (E1_… hoặc Table 1). Hỏi nhóm trước khi chạy."

## 7. Chấm lại 50 ứng viên của mỗi prompt (M = 4 lần gọi ImageReward mỗi ứng viên)
Decode và chấm giống hệt `lookahead_sampling.py`: VAE của SD 1.5 (fp16), `latents / scaling_factor`, `image_processor.postprocess` ra PIL, rồi `do_eval(..., ["ImageReward"])`. Điểm được lưu lên Drive sau mỗi 25 prompt; bị ngắt thì Run all lại là chạy tiếp.

In [ ]:
import io as _io
from PIL import Image, ImageFilter, ImageOps
from diffusers import AutoencoderKL
from diffusers.image_processor import VaeImageProcessor
from fks_utils import do_eval

vae = AutoencoderKL.from_pretrained(MODEL_NAME, subfolder="vae", torch_dtype=torch.float16).to("cuda").eval()
img_proc = VaeImageProcessor(vae_scale_factor=2 ** (len(vae.config.block_out_channels) - 1))
DEC_BATCH = 25


@torch.inference_mode()
def decode(lat):
    """[n, 4, 64, 64] latents of the pool -> list of n PIL images (as lookahead_sampling.py decodes them)."""
    out = []
    for s in range(0, lat.shape[0], DEC_BATCH):
        z = lat[s:s + DEC_BATCH].to("cuda", torch.float16) / vae.config.scaling_factor
        out += img_proc.postprocess(vae.decode(z, return_dict=False)[0], output_type="pil")
    return out


def _jpeg(img, q=75):
    buf = _io.BytesIO()
    img.save(buf, format="JPEG", quality=q)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def aug_views(img, rng):
    """4 nhẹ, cố định: lật ngang; crop 90% (vị trí ngẫu nhiên, rng cố định) rồi resize lại; JPEG q = 75; Gaussian blur r = 1."""
    w, h = img.size
    cw, ch = int(round(0.9 * w)), int(round(0.9 * h))
    x0, y0 = int(rng.integers(0, w - cw + 1)), int(rng.integers(0, h - ch + 1))
    crop = img.crop((x0, y0, x0 + cw, y0 + ch)).resize((w, h), Image.BICUBIC)
    return [ImageOps.mirror(img), crop, _jpeg(img), img.filter(ImageFilter.GaussianBlur(radius=1.0))]


def score_prompt(i):
    """Điểm F1 của 50 ứng viên prompt i, cùng thứ tự latent với pool gốc."""
    lat, _ = load_pool(RAW_DIR, i)
    prompt = META[i]["prompt"]
    if VARIANT == "aug":
        rng = np.random.default_rng([F1_SEED, i])
        views = [v for img in decode(lat) for v in aug_views(img, rng)]            # [K * M], ứng viên k ở k*M..k*M+M-1
    else:
        g = torch.Generator().manual_seed(F1_SEED * 100003 + i)
        noise = torch.randn((M_F1, *lat.shape), generator=g, dtype=torch.float32)  # [M, K, 4, 64, 64]
        noisy = (lat.float().unsqueeze(0) + SIGMA_Z * noise).transpose(0, 1).reshape(-1, *lat.shape[1:])
        views = decode(noisy)                                                       # [K * M], cùng thứ tự
    r = np.asarray(do_eval(prompt=[prompt] * len(views), images=views, metrics_to_compute=["ImageReward"])
                   ["ImageReward"]["result"], dtype=np.float64)
    assert r.shape[0] == lat.shape[0] * M_F1
    return r.reshape(lat.shape[0], M_F1).mean(1)


score_path = f"{EXP_DRIVE}/F1_scores_{RUN_TAG}.json"
SCORES = {int(k): v for k, v in json.load(open(score_path)).items()} if os.path.exists(score_path) else {}
todo = [i for i in PROMPT_IDS if i not in SCORES]
for n, i in enumerate(tqdm(todo, desc=f"Chấm {RUN_TAG}")):
    SCORES[i] = score_prompt(i).tolist()
    if n % 25 == 24:
        json.dump({str(k): v for k, v in SCORES.items()}, open(score_path, "w"))
json.dump({str(k): v for k, v in SCORES.items()}, open(score_path, "w"))
print(f"✅ Đã chấm {len(SCORES)} prompt → {score_path}")
del vae
torch.cuda.empty_cache()

## 8. Pool đã chấm lại (cùng latent) và Phase 2 (seed 100, phải chạy một mạch)

In [ ]:
F1_DIR = write_derived(RAW_DIR, f"{LOCAL_DERIVED}/{RUN_TAG}", lambda i, res: SCORES[i])
print("=" * 90, f"\n▶ {RUN_LABEL}")
RUNS[RUN_LABEL] = phase2(F1_DIR, f"F1_{RUN_TAG}_{MODEL_TAG}_n{K}_M{M_F1}_DDPM100_lmbda{REF_LAMBDA}_seed{SEED_P2}{SUBSET_TAG}")

# Dùng luôn run F1 đã chạy xong ở notebook F1 còn lại, để bảng cuối đủ cả hai biến thể
KNOWN = {"LiDAR-Aug (M=4)": f"F1_Aug_{MODEL_TAG}_n{K}_M{M_F1}_DDPM100_lmbda{REF_LAMBDA}_seed{SEED_P2}",
         "LiDAR-LatentSmooth σz=0.3 (M=4)": f"F1_Latent_sz0.3_{MODEL_TAG}_n{K}_M{M_F1}_DDPM100_lmbda{REF_LAMBDA}_seed{SEED_P2}"}
for lab, name in KNOWN.items():
    if not RUNS.get(lab):
        d = find_target([name])
        if d and ref_ok(d, None):
            RUNS[lab] = d
            print(f"↺ Dùng run đã có trên Drive: {lab} ← {name}")
print(RUNS)

## 9. GenEval và bảng kết quả

In [ ]:
# Dùng lại cache GenEval của hai run mốc mà E3 đã chấm (cùng bộ chấm, cùng run), đỡ chấm lại
_e3_cache = f"{DRIVE_DIR}/extra_experiments/E3_compute_matched/geneval_cache"
os.makedirs(f"{EXP_DRIVE}/geneval_cache", exist_ok=True)
for _d in RUNS.values():
    _f = f"{os.path.basename(os.path.normpath(_d))}.json"
    if os.path.exists(f"{_e3_cache}/{_f}") and not os.path.exists(f"{EXP_DRIVE}/geneval_cache/{_f}"):
        shutil.copy(f"{_e3_cache}/{_f}", f"{EXP_DRIVE}/geneval_cache/{_f}")

In [ ]:
# GenEval chính thức (Mask2Former Swin-S + OpenCLIP ViT-L-14): phần nạp mô hình và các hàm
# color_classification / relative_position / evaluate_single_image được chép nguyên văn từ cell 16
# của notebook LiDAR_Table2_Replication_Colab.ipynb; chỉ thêm phần chạy theo batch và cache theo prompt.
import os, glob, json, time
import numpy as np, pandas as pd
from PIL import Image
import torch
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

device = "cuda" if torch.cuda.is_available() else "cpu"

mask2former_id = "facebook/mask2former-swin-small-coco-instance"
image_processor = AutoImageProcessor.from_pretrained(mask2former_id)
image_processor.size = {'shortest_edge': 800, 'longest_edge': 1333}  # Chuẩn bài báo GenEval / MMDetection COCO gốc (800px)
detector = Mask2FormerForUniversalSegmentation.from_pretrained(mask2former_id).to(device).eval()
id2label = detector.config.id2label

# Danh sách 80 lớp COCO chuẩn bài báo GenEval (object_names.txt)
COCO_CLASSES_80 = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
    "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
    "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
    "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
    "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
    "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "computer mouse", "tv remote",
    "computer keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book",
    "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush"
]

# 5. Nạp mô hình OpenCLIP ViT-L-14 (Chuẩn bài báo GenEval gốc 100%)
print("\n📦 Đang nạp mô hình OpenCLIP ViT-L-14 (openai weights) cho kiểm tra màu sắc (colors & color_attr)...\n")
try:
    import open_clip
    clip_arch = "ViT-L-14"
    clip_model, _, clip_transform = open_clip.create_model_and_transforms(clip_arch, pretrained="openai", device=device)
    clip_tokenizer = open_clip.get_tokenizer(clip_arch)
    clip_model.eval()
    use_open_clip = True
    print("✅ Đã nạp thành công OpenCLIP ViT-L-14 chuẩn gốc!")
except Exception as e:
    print(f"⚠️ Không nạp được open_clip ({e}), chuyển sang dự phòng Hugging Face CLIP ViT-L/14...")
    from transformers import CLIPModel, CLIPProcessor
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
    clip_transform = lambda img: clip_processor(images=img, return_tensors="pt")["pixel_values"][0]
    clip_tokenizer = None
    use_open_clip = False

COLORS = ['red', 'orange', 'yellow', 'green', 'blue', 'purple', 'pink', 'brown', 'black', 'white']
COLOR_CLASSIFIERS = {}

def get_color_classifier(classname):
    """Xây dựng và cache zero-shot color classifier chuẩn GenEval cho từng classname"""
    if classname not in COLOR_CLASSIFIERS:
        c_name = classname if classname else "object"
        templates = [
            f"a photo of a {{c}} {c_name}",
            f"a photo of a {{c}}-colored {c_name}",
            f"a photo of a {{c}} object"
        ]
        with torch.inference_mode():
            if use_open_clip:
                zeroshot_weights = []
                for c in COLORS:
                    prompts = [t.format(c=c) for t in templates]
                    tokens = clip_tokenizer(prompts).to(device)
                    text_embeddings = clip_model.encode_text(tokens)
                    text_embeddings = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)
                    class_embedding = text_embeddings.mean(dim=0)
                    class_embedding = class_embedding / class_embedding.norm(dim=-1, keepdim=True)
                    zeroshot_weights.append(class_embedding)
                COLOR_CLASSIFIERS[classname] = torch.stack(zeroshot_weights, dim=1)  # (D, 10)
            else:
                prompts = [t.format(c=c) for c in COLORS for t in templates]
                inputs = clip_processor(text=prompts, return_tensors="pt", padding=True).to(device)
                text_out = clip_model.get_text_features(**inputs)
                text_out = text_out / text_out.norm(dim=-1, keepdim=True)
                COLOR_CLASSIFIERS[classname] = text_out.view(len(COLORS), 3, -1).mean(dim=1).T  # (D, 10)
    return COLOR_CLASSIFIERS[classname]

def color_classification(image, objects, classname):
    """Phân loại màu sắc chuẩn 100% official GenEval: Ghép mask nền xám #999 + CLIP ViT-L-14"""
    if not objects:
        return []
    clf = get_color_classifier(classname)
    blank = Image.new("RGB", image.size, color="#999")
    crop_tensors = []
    for obj in objects:
        box, mask = obj[0], obj[1] if len(obj) > 1 else None
        x1, y1, x2, y2 = int(max(0, box[0])), int(max(0, box[1])), int(min(image.width, box[2])), int(min(image.height, box[3]))
        if (x2 - x1 <= 12 or y2 - y1 <= 12):
            crop_tensors.append(clip_transform(Image.new("RGB", (224, 224), color="#999")))
            continue
        if mask is not None:
            comp = Image.composite(image, blank, Image.fromarray((mask > 0).astype(np.uint8) * 255))
        else:
            comp = image
        crop = comp.crop((x1, y1, x2, y2))
        crop_tensors.append(clip_transform(crop))

    crop_batch = torch.stack(crop_tensors).to(device)
    with torch.inference_mode():
        if use_open_clip:
            img_features = clip_model.encode_image(crop_batch)
        else:
            img_features = clip_model.get_image_features(pixel_values=crop_batch)
        img_features = img_features / img_features.norm(dim=-1, keepdim=True)
        logits = 100.0 * (img_features @ clf)
        pred_indices = logits.argmax(dim=-1).cpu().tolist()
        return [COLORS[idx] for idx in pred_indices]

def relative_position(obj_a, obj_b):
    """Tính vị trí tương quan của A đối với B chuẩn 100% bài báo GenEval gốc (evaluate_images.py:relative_position)"""
    box_a = obj_a[0] if isinstance(obj_a, (tuple, list)) else obj_a
    box_b = obj_b[0] if isinstance(obj_b, (tuple, list)) else obj_b
    boxes = np.array([box_a[:4], box_b[:4]])[:, :4].reshape(2, 2, 2)
    center_a, center_b = boxes.mean(axis=-2)
    dim_a, dim_b = np.abs(np.diff(boxes, axis=-2))[..., 0, :]
    offset = center_a - center_b
    revised_offset = np.maximum(np.abs(offset) - 0.1 * (dim_a + dim_b), 0) * np.sign(offset)
    if np.all(np.abs(revised_offset) < 1e-3):
        return set()
    norm_offset = np.linalg.norm(offset)
    if norm_offset <= 1e-5:
        return set()
    dx, dy = revised_offset / norm_offset
    relations = set()
    if dx < -0.5: relations.add("left of")
    if dx > 0.5: relations.add("right of")
    if dy < -0.5: relations.add("above")
    if dy > 0.5: relations.add("below")
    return relations

def evaluate_single_image(image, objects, metadata):
    """Đánh giá 1 ảnh chuẩn 100% theo official GenEval (evaluate_images.py:evaluate)"""
    correct = True
    matched_groups = []
    # 1. Kiểm tra các vật thể được yêu cầu (include)
    for req in metadata.get('include', []):
        classname = req['class'].lower()
        matched = True
        found_objects = objects.get(classname, [])[:req['count']]
        if len(found_objects) < req['count']:
            correct = matched = False
        else:
            if 'color' in req:
                colors = color_classification(image, found_objects, classname)
                if colors.count(req['color'].lower()) < req['count']:
                    correct = matched = False
            if 'position' in req and matched:
                expected_rel, target_group = req['position']
                if target_group >= len(matched_groups) or matched_groups[target_group] is None:
                    correct = matched = False
                else:
                    for obj in found_objects:
                        for target_obj in matched_groups[target_group]:
                            true_rels = relative_position(obj, target_obj)
                            if expected_rel not in true_rels:
                                correct = matched = False
                                break
                        if not matched:
                            break
        if matched:
            matched_groups.append(found_objects)
        else:
            matched_groups.append(None)
    # 2. Kiểm tra các vật thể bị loại trừ (exclude - đặc biệt quan trọng cho task counting)
    for req in metadata.get('exclude', []):
        classname = req['class'].lower()
        if len(objects.get(classname, [])) >= req['count']:
            correct = False
    return correct



assert use_open_clip, ("open_clip không nạp được; cell này sẽ chấm màu bằng HF CLIP, khác bộ chấm của Table 1. "
                      "Sửa cài đặt open_clip rồi chạy lại.")
TASKS = ['single_object', 'two_object', 'counting', 'colors', 'position', 'color_attr']


def _detect_batch(images, tag):
    conf_thr = 0.9 if tag == 'counting' else 0.3
    image_processor.size = {'shortest_edge': 800, 'longest_edge': 1333}
    inputs = image_processor(images=images, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = detector(**inputs)
    results = image_processor.post_process_instance_segmentation(
        outputs, target_sizes=[im.size[::-1] for im in images], threshold=conf_thr)
    all_detected = []
    for res in results:
        segmentation = res["segmentation"].detach().cpu().numpy()
        segments_info = sorted(res["segments_info"], key=lambda s: s.get("score", 0.0), reverse=True)
        detected = {}
        for seg in segments_info:
            lbl_id = seg["label_id"]
            c_name = COCO_CLASSES_80[lbl_id].lower() if (0 <= lbl_id < len(COCO_CLASSES_80)) else id2label.get(lbl_id, "unknown").lower()
            mask = (segmentation == seg["id"])
            y_indices, x_indices = np.where(mask)
            if len(x_indices) > 0 and len(y_indices) > 0:
                box = [float(np.min(x_indices)), float(np.min(y_indices)), float(np.max(x_indices)), float(np.max(y_indices)),
                       seg.get("score", 1.0)]
                if len(detected.setdefault(c_name, [])) < 16:
                    detected[c_name].append((box, mask))
        all_detected.append(detected)
    return all_detected


def geneval_correct(images, meta, batch_size=16):
    """Đúng/sai (1.0/0.0) theo GenEval cho từng ảnh PIL của cùng một prompt."""
    tag = meta.get('tag', 'single_object')
    out = []
    for s in range(0, len(images), batch_size):
        chunk = images[s:s + batch_size]
        for img, det in zip(chunk, _detect_batch(chunk, tag)):
            out.append(1.0 if evaluate_single_image(img, det, meta) else 0.0)
    return out


def geneval_score_dir(target_dir, overwrite=False):
    """Điểm GenEval theo prompt (trung bình 4 ảnh) cho một thư mục Target.

    Cache để trong thư mục của thí nghiệm (không ghi vào thư mục run Table 1 dùng chung), kèm tên backend CLIP.
    """
    os.makedirs(f"{EXP_DRIVE}/geneval_cache", exist_ok=True)
    cache = f"{EXP_DRIVE}/geneval_cache/{os.path.basename(os.path.normpath(target_dir))}.json"
    per = {}
    # Dấu của run: thời điểm tạo args.json. Thư mục run bị xoá rồi tạo lại (run dở dang) thì dấu đổi, cache cũ bị bỏ.
    args_f = f"{target_dir}/args.json"
    stamp = str(round(os.path.getmtime(args_f))) if os.path.exists(args_f) else None
    if os.path.exists(cache) and not overwrite:
        blob = json.load(open(cache))
        assert blob.get("backend") == "open_clip", f"cache {cache} dùng backend khác: {blob.get('backend')}"
        if blob.get("stamp") == stamp:
            per = {int(k): v for k, v in blob["per_prompt"].items()}
        else:
            print(f"⚠️ Cache GenEval của {os.path.basename(target_dir)} thuộc một lần chạy khác -> chấm lại.")
    todo = [i for i in PROMPT_IDS if i not in per and glob.glob(f"{target_dir}/{i:05d}/samples/*.png")]
    for n, i in enumerate(tqdm(todo, desc=f"GenEval {os.path.basename(target_dir)[:40]}")):
        ims = [Image.open(p).convert("RGB") for p in sorted(glob.glob(f"{target_dir}/{i:05d}/samples/*.png"))]
        per[i] = float(np.mean(geneval_correct(ims, META[i])))
        if n % 50 == 49:
            json.dump({"backend": "open_clip", "stamp": stamp, "per_prompt": {str(k): v for k, v in per.items()}}, open(cache, "w"))
    if todo:
        json.dump({"backend": "open_clip", "stamp": stamp, "per_prompt": {str(k): v for k, v in per.items()}}, open(cache, "w"))
    return per


def geneval_overall(per, ids=None):
    """Overall GenEval = trung bình của 6 điểm theo task (giống cell 16 và bài báo)."""
    ids = [i for i in (per.keys() if ids is None else ids) if i in per]
    means = [np.mean([per[i] for i in ids if META[i].get('tag') == t]) for t in TASKS
             if any(META[i].get('tag') == t for i in ids)]
    return float(np.mean(means))

print("✅ GenEval sẵn sàng.")

In [ ]:
METRICS = ["ImageReward", "Clip-Score", "HumanPreference", "AS", "GenEval"]
SHORT = {"ImageReward": "IR", "Clip-Score": "CLIP", "HumanPreference": "HPS", "AS": "AS", "GenEval": "GenEval"}


def per_prompt(target_dir):
    out = {m: {} for m in METRICS}
    for i in PROMPT_IDS:
        f = f"{target_dir}/{i:05d}/results.json"
        if os.path.exists(f):
            r = json.load(open(f))
            for m in METRICS[:-1]:
                if m in r and isinstance(r[m], dict) and "mean" in r[m]:
                    out[m][i] = float(r[m]["mean"])
    out["GenEval"] = geneval_score_dir(target_dir)
    return out


def summarize(runs, ref, csv_name, n_boot=2000):
    """Bảng trung bình và chênh lệch so với `ref` (bootstrap theo prompt, 95% CI)."""
    runs = {k: v for k, v in runs.items() if v}
    data = {lab: per_prompt(d) for lab, d in runs.items()}
    ids = sorted(set.intersection(*[set(v["ImageReward"]) & set(v["GenEval"]) for v in data.values()]))
    tags = np.array([META[i].get("tag") for i in ids])
    task_masks = [tags == t for t in TASKS if (tags == t).any()]
    B = np.random.default_rng(0).integers(0, len(ids), size=(n_boot, len(ids)))

    def level(v, idx):
        if idx.ndim == 1:
            idx = idx[None]
        if metric == "GenEval":
            vals = v[idx]
            return np.mean([(vals * mk[idx]).sum(1) / np.maximum(mk[idx].sum(1), 1) for mk in task_masks], axis=0)
        return v[idx].mean(1)

    rows = []
    for lab in runs:
        row = {"run": lab, "n_prompts": len(ids)}
        for metric in METRICS:
            v = np.array([data[lab][metric].get(i, np.nan) for i in ids])
            r = np.array([data[ref][metric].get(i, np.nan) for i in ids])
            row[SHORT[metric]] = float(level(v, np.arange(len(ids)))[0])
            if lab != ref:
                d_boot = level(v, B) - level(r, B)
                d = row[SHORT[metric]] - float(level(r, np.arange(len(ids)))[0])
                lo, hi = np.percentile(d_boot, [2.5, 97.5])
                star = "*" if (lo > 0 or hi < 0) else ""
                row[f"Δ{SHORT[metric]} [95% CI]"] = f"{d:+.4f} [{lo:+.4f}, {hi:+.4f}]{star}"
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(f"{EXP_DRIVE}/{csv_name}", index=False)
    pd.set_option("display.width", 250)
    pd.set_option("display.max_columns", 30)
    print(f"Δ = run − {ref}; * = 95% CI không chứa 0. Đã lưu {EXP_DRIVE}/{csv_name}")
    display(df)
    return df

In [ ]:
import re


def _ci(cell):
    """'+0.0123 [-0.0030, +0.0280]*' -> (0.0123, -0.0030, 0.0280)."""
    m = re.match(r"\s*([+-]?\d+\.\d+)\s*\[\s*([+-]?\d+\.\d+)\s*,\s*([+-]?\d+\.\d+)\s*\]", str(cell))
    return tuple(float(x) for x in m.groups()) if m else None


def verdict(df, other_label):
    """RS-LiDAR so với dòng `other_label` trong bảng có mốc là RS-LiDAR (Δ = other − RS-LiDAR); tiêu chí một seed như E3b.

    Thua:  dòng kia cao hơn có * ở IR hoặc GenEval.
    Thắng: RS-LiDAR >= dòng kia theo điểm ước lượng ở cả IR và GenEval, và dòng kia không cao hơn có * ở IR/CLIP/HPS/GenEval.
    Hoà:   còn lại.
    """
    row = df[df["run"] == other_label]
    if row.empty:
        return None
    cis = {m: _ci(row.iloc[0].get(f"Δ{m} [95% CI]")) for m in ("IR", "CLIP", "HPS", "GenEval")}
    ir, ge = cis["IR"], cis["GenEval"]
    if ir is None or ge is None:
        return None
    if ir[1] > 0 or ge[1] > 0:
        return "Thua"
    if ir[0] <= 0 and ge[0] <= 0 and not any(v is not None and v[1] > 0 for v in cis.values()):
        return "Thắng"
    return "Hoà"

In [ ]:
df_f1 = summarize(RUNS, ref="RS-LiDAR K=50 M=4", csv_name="F1_summary.csv")
CALLS = {"RS-LiDAR K=50 M=4": 4, "LiDAR K=50": 1}
lines = []
for lab in RUNS:
    if lab.startswith("LiDAR-"):
        CALLS[lab] = 4
        lines.append(f"- RS-LiDAR so với {lab}: **{verdict(df_f1, lab)}**")
m = df_f1[["run", "IR", "CLIP", "HPS", "GenEval"]].copy()
m.insert(1, "Lần gọi reward / ứng viên", m["run"].map(CALLS))
m = m.round(3)
text = ("# F1: các chỉ số (SD 1.5, DDPM-100, 553 GenEval prompt, seed 100, cùng 50 ứng viên)\n\n"
        + m.to_markdown(index=False) + "\n\n" + "\n".join(lines)
        + "\n\nKết luận dùng tiêu chí một seed như E3b (README, thẻ T11). Δ trong `F1_summary.csv` = run − RS-LiDAR.\n")
open(f"{EXP_DRIVE}/F1_metrics.md", "w", encoding="utf-8").write(text)
print(text)

## Tự ngắt máy khi chạy xong
Cell cuối tự ngắt runtime Colab sau khi Run all xong (tiết kiệm compute unit). Kết quả đã lưu trên Drive.

In [ ]:
# Tự ngắt runtime khi Run all chạy xong, để không tốn thêm compute unit.
# Kết quả đã nằm trên Drive; đẩy hết dữ liệu đang chờ ghi lên Drive trước khi ngắt.
# Muốn giữ máy để xem/chạy tiếp thì đặt AUTO_DISCONNECT = False trước khi Run all.
AUTO_DISCONNECT = True
if AUTO_DISCONNECT:
    try:
        from google.colab import drive, runtime
        drive.flush_and_unmount()
        print("🛑 Đã chạy xong và lưu lên Drive. Tự ngắt runtime Colab.")
        runtime.unassign()
    except Exception as e:
        print("⚠️ Không tự ngắt được runtime, hãy tự ngắt: Runtime → Disconnect and delete runtime.", e)